# SunSim at the PoET exposures (poet.dag)

In [ ]:
import glob, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

POET_OUT = '/data/spotless/scratch/sstucki/starsim/poet'
SETS = os.path.join(POET_OUT, 'sets')
RES = os.path.join(POET_OUT, 'results')
TABLE = os.path.join(RES, 'poet_simulations.csv')
NPY = os.path.join(RES, 'poet_simulations.npy')
META = ['set', 'noise_th', 'I_th', 'A_th', 'T_sp', 'instrument', 'mask']
OBSERVABLES = ['rv', 'fwhm', 'bis', 'contrast', 'lc']

## Load

In [ ]:
if os.path.exists(TABLE):
    table = pd.read_csv(TABLE)
else:                                            # poet_table.sub not run yet: the simulations done so far
    parts = []
    for folder in sorted(glob.glob(os.path.join(SETS, '*'))):
        path = os.path.join(folder, 'exposures_simulated.csv')
        if not os.path.exists(path):
            print('no exposures_simulated.csv:', os.path.basename(folder))
            continue
        d = pd.read_csv(path)
        for k, v in json.load(open(os.path.join(folder, 'poet_run.json'))).items():
            d[k] = v
        parts.append(d)
    table = pd.concat(parts, ignore_index=True)


def case_key(r):
    return 'I_th_%.2f_A_th_%g_T_%d_%s_%s' % (r['I_th'], r['A_th'], r['T_sp'], r['instrument'], r['mask'])


table['case'] = table.apply(case_key, axis=1)
CASES = sorted(table['case'].unique())
table['time'] = pd.to_datetime(table['jd_utc'] - 2440587.5, unit='D')
for c in CASES:
    d = table[table['case'] == c]
    print('%-45s %5d exposures, %5d with an epoch' % (c, len(d), d['epoch'].ge(0).sum()))

## npy file

In [ ]:
results = {}
for c in CASES:
    d = table[table['case'] == c].sort_values('jdb')
    r = {k: d[k].to_numpy() for k in d.columns if k not in META + ['case', 'time']}
    r.update({k: d[k].iloc[0] for k in META})
    results[c] = r
np.save(NPY, results, allow_pickle=True)
print(NPY)
print(list(results[CASES[0]]))

In [ ]:
check = np.load(NPY, allow_pickle=True).item()
assert sorted(check) == CASES

## Standard deviation over the exposures

In [ ]:
std = table.groupby('case')[OBSERVABLES + ['ff_sp', 'ff_fc']].std()
std

## Time series

In [ ]:
ok = table['epoch'] >= 0
styles = {('None', 'ESPRESSO_SUN'): ('C0', '-'), ('None', 'PoET_noflag'): ('C1', '-'),
          ('PoET', 'ESPRESSO_SUN'): ('C0', '--'), ('PoET', 'PoET_noflag'): ('C1', '--')}
sets = sorted(table['set'].unique())
fig, axes = plt.subplots(len(OBSERVABLES), len(sets), figsize=(8 * len(sets), 2.6 * len(OBSERVABLES)),
                         sharex=True, sharey='row', squeeze=False)
for j, s in enumerate(sets):
    for (inst, mask), (color, ls) in styles.items():
        d = table[ok & (table['set'] == s) & (table['instrument'] == inst) & (table['mask'] == mask)]
        if d.empty:
            continue
        for i, q in enumerate(OBSERVABLES):
            axes[i, j].plot(d['time'], d[q], '.', ms=2, color=color, alpha=0.6 if ls == '-' else 0.3,
                            label='%s, %s' % (inst, mask))
    axes[0, j].set_title(s)
for i, q in enumerate(OBSERVABLES):
    axes[i, 0].set_ylabel(q)
axes[0, 0].legend(markerscale=5, fontsize=8)
fig.autofmt_xdate()
plt.tight_layout()

## Filling factors

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
for j, s in enumerate(sets):
    d = table[ok & (table['case'] == table[table['set'] == s]['case'].iloc[0])]
    axes[0].plot(d['time'], d['ff_sp'], '.', ms=2, color='C%d' % j, label=s)
    axes[1].plot(d['time'], d['ff_fc'], '.', ms=2, color='C%d' % j)
axes[0].set_ylabel('ff_sp [%]')
axes[1].set_ylabel('ff_fc [%]')
axes[0].legend(markerscale=5)
plt.tight_layout()

## PoET instrument minus no instrument

In [ ]:
fig, axes = plt.subplots(len(OBSERVABLES), 1, figsize=(16, 2.6 * len(OBSERVABLES)), sharex=True)
k = 0
for s in sets:
    for mask in sorted(table['mask'].unique()):
        a = table[ok & (table['set'] == s) & (table['mask'] == mask) & (table['instrument'] == 'None')]
        b = table[ok & (table['set'] == s) & (table['mask'] == mask) & (table['instrument'] == 'PoET')]
        if a.empty or b.empty:
            continue
        m = a.merge(b, on='jdb', suffixes=('_none', '_poet'))
        for i, q in enumerate(OBSERVABLES):
            axes[i].plot(m['time_none'], m[q + '_poet'] - m[q + '_none'], '.', ms=2, color='C%d' % k,
                         label='%s, %s' % (s, mask))
        k += 1
for i, q in enumerate(OBSERVABLES):
    axes[i].set_ylabel('d' + q)
axes[0].legend(markerscale=5, fontsize=8)
plt.tight_layout()

## RV against the other observables

In [ ]:
others = [q for q in OBSERVABLES if q != 'rv'] + ['ff_sp', 'ff_fc']
fig, axes = plt.subplots(len(CASES), len(others), figsize=(3.2 * len(others), 2.8 * len(CASES)), squeeze=False)
for i, c in enumerate(CASES):
    d = table[ok & (table['case'] == c)]
    for j, q in enumerate(others):
        ax = axes[i, j]
        ax.plot(d[q], d['rv'], '.', ms=1, color='k', alpha=0.4)
        rho = np.corrcoef(d[q], d['rv'])[0, 1]
        ax.set_title('r = %.2f' % rho, fontsize=9)
        if i == len(CASES) - 1:
            ax.set_xlabel(q)
    axes[i, 0].set_ylabel('rv\n' + c, fontsize=7)
plt.tight_layout()